<a href="https://colab.research.google.com/github/yolacin/LocalDispersion/blob/main/ToGithub_AnalysisAndVisualizationSuite.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pickle
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import drive

drive.mount('/content/drive')

# ==========================================
# 0. SETUP AND DATA LOADING
# ==========================================
save_dir = "save_dir" # save directory string to be placed in.
st_num_list = [773535, 773422, 760384, 718126, 716241]
st_num = st_num_list[4]
file_path = os.path.join(save_dir, "Station_{}_Complete_Results.pkl".format(st_num))

save_dir_after_reload = "save_dir_after_reload" # save directory after reload to be placed in.

with open(file_path, 'rb') as f:
    data = pickle.load(f)

y_test = data['targets']['y_test']
preds = data['predictions']
vars_ = data['variances']
station = data['metadata']['station']

def get_metrics(y_true, y_pred):
    mae = np.mean(np.abs(y_true - y_pred))
    # Add epsilon to prevent division by zero in MAPE
    mape = np.mean(np.abs((y_true - y_pred) / (y_true + 1e-8))) * 100
    return mae, mape

In [ ]:
# ==========================================
# TASK 1: KNN Short/Long Combinations -> LaTeX
# ==========================================
print("\n--- TASK 1: KNN Combinations LaTeX Table ---")
# This is a standard string, so single braces are fine here
latex_str_t1 = "\\begin{table}[h]\n\\centering\n\\begin{tabular}{lcc}\n\\hline\nModel & MAE & MAPE (\\%) \\\\\n\\hline\n"

# a) KNN Short and KNN Long
mae_s, mape_s = get_metrics(y_test, preds['knn_short'])
mae_l, mape_l = get_metrics(y_test, preds['knn_long'])
latex_str_t1 += f"KNN Short & {mae_s:.3f} & {mape_s:.3f} \\\\\n"
latex_str_t1 += f"KNN Long & {mae_l:.3f} & {mape_l:.3f} \\\\\n"

# b) Alpha-Weighted KNNs
alphas = np.arange(0.1, 1.0, 0.1)
for a in alphas:
    p_weight = a * preds['knn_short'] + (1 - a) * preds['knn_long']
    m, mp = get_metrics(y_test, p_weight)
    latex_str_t1 += f"Weighted $\\alpha={a:.1f}$ & {m:.3f} & {mp:.3f} \\\\\n"

# c) Variance Weighted KNNs
eps = 1e-8
v1, v2 = vars_['knn_short_var'], vars_['knn_long_var']
w1 = v1 / (v1 + v2 + eps)
w2 = v2 / (v1 + v2 + eps)
p_var_weight = w1 * preds['knn_short'] + w2 * preds['knn_long']
preds['knn_variance'] = p_var_weight # Save for Task 4
m_v, mp_v = get_metrics(y_test, p_var_weight)

# FIX: Double curly braces used for {{tabular}} and {{table}} inside the f-string
latex_str_t1 += f"Variance-Weighted & {m_v:.3f} & {mp_v:.3f} \\\\\n\\hline\n\\end{{tabular}}\n\\end{{table}}"

print(latex_str_t1)


--- TASK 1: KNN Combinations LaTeX Table ---
\begin{table}[h]
\centering
\begin{tabular}{lcc}
\hline
Model & MAE & MAPE (\%) \\
\hline
KNN Short & 58.669 & 6.325 \\
KNN Long & 57.694 & 6.237 \\
Weighted $\alpha=0.1$ & 56.520 & 6.108 \\
Weighted $\alpha=0.2$ & 55.590 & 6.005 \\
Weighted $\alpha=0.3$ & 54.925 & 5.931 \\
Weighted $\alpha=0.4$ & 54.563 & 5.890 \\
Weighted $\alpha=0.5$ & 54.506 & 5.883 \\
Weighted $\alpha=0.6$ & 54.749 & 5.907 \\
Weighted $\alpha=0.7$ & 55.330 & 5.968 \\
Weighted $\alpha=0.8$ & 56.182 & 6.059 \\
Weighted $\alpha=0.9$ & 57.324 & 6.181 \\
Variance-Weighted & 55.161 & 5.981 \\
\hline
\end{tabular}
\end{table}


In [ ]:

# ==========================================
# TASK 2 & 3: Scatter Plots of Errors & Variances
# ==========================================
E1 = np.abs(y_test - preds['knn_short'])
E2 = np.abs(y_test - preds['knn_long'])
V1 = vars_['knn_short_var']
V2 = vars_['knn_long_var']

def save_scatter(x, y, xlabel, ylabel, filename):
    plt.figure(figsize=(6, 5))
    plt.scatter(x, y, alpha=0.5, s=10)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(f"{xlabel} vs {ylabel}")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir_after_reload, filename), dpi=300)
    plt.close()

save_scatter(E1, E2, "Error KNN Short (E1)", "Error KNN Long (E2)", f"Task2_Scatter_E1_E2_{station}.png")
save_scatter(V1, E1, "Variance KNN Short (V1)", "Error KNN Short (E1)", f"Task3_Scatter_V1_E1_{station}.png")
save_scatter(V2, E2, "Variance KNN Long (V2)", "Error KNN Long (E2)", f"Task3_Scatter_V2_E2_{station}.png")

In [ ]:
# ==========================================
# HELPER FOR TASKS 4, 5, 8, 9 (Volatile selection)
# ==========================================
thresholds = np.arange(0.70, 1.00, 0.01)

def eval_volatile(target_preds, dispersion_measure, thresh_list, metric='mae'):
    scores = []
    # Using KNNC2 var to rank volatility
    ranked_indices = np.argsort(dispersion_measure)

    for th in thresh_list:
        # Top (1-th) fraction are unstable
        unstable_count = int((1 - th) * len(y_test))
        if unstable_count == 0:
            unstable_count = 1
        unstable_idx = ranked_indices[-unstable_count:]

        y_t = y_test[unstable_idx]
        y_p = target_preds[unstable_idx]

        if metric == 'mae':
            scores.append(np.mean(np.abs(y_t - y_p)))
        else: # mape
            scores.append(np.mean(np.abs((y_t - y_p) / (y_t + 1e-8))) * 100)
    return scores

In [ ]:


# ==========================================
# TASK 4 & 5: KNN Variants MAE & MAPE vs Threshold
# ==========================================
models_t4 = {
    'KNN Short': preds['knn_short'],
    'KNN Long': preds['knn_long'],
    'KNN Average': preds['knn_avg'],
    'KNN Variance': preds['knn_variance']
}

def plot_thresholds(models_dict, metric, filename):
    plt.figure(figsize=(10, 6))
    for name, p in models_dict.items():
        scores = eval_volatile(p, vars_['knnc2_var'], thresholds, metric=metric)
        plt.plot(thresholds, scores, marker='.', label=name)

    plt.xlabel("Rejection Threshold")
    plt.ylabel(f"{metric.upper()} on Unstable Data")
    plt.title(f"{metric.upper()} Progression on Volatile Data - Station {station}")
    plt.legend()
    plt.grid(True, alpha=0.4)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir_after_reload, filename), dpi=300)
    plt.close()

plot_thresholds(models_t4, 'mae', f"Task4_KNN_Variants_MAE_{station}.png")
plot_thresholds(models_t4, 'mape', f"Task5_KNN_Variants_MAPE_{station}.png")

In [ ]:


# ==========================================
# TASK 6 & 7: Macro Models LaTeX Tables
# ==========================================
def print_model_latex(models_dict, title):
    latex_str = f"\\begin{{table}}[h]\n\\caption{{{title}}}\n\\centering\n\\begin{{tabular}}{{lcc}}\n\\hline\nModel & MAE & MAPE (\\%) \\\\\n\\hline\n"
    for name, p in models_dict.items():
        m, mp = get_metrics(y_test, p)
        latex_str += f"{name} & {m:.3f} & {mp:.3f} \\\\\n"
    latex_str += "\\hline\n\\end{tabular}\n\\end{table}\n"
    print(latex_str)

models_no_knn = {'KNN Avg': preds['knn_avg'], 'LSTM (No KNN)': preds['lstm_no_knn'], 'DNN (No KNN)': preds['dnn_no_knn'], 'ARIMA (No KNN)': preds['arima_no_knn']}
models_with_knn = {'KNN Avg': preds['knn_avg'], 'LSTM (With KNN)': preds['lstm_with_knn'], 'DNN (With KNN)': preds['dnn_with_knn'], 'ARIMA (With KNN)': preds['arima_with_knn']}

print("\n--- TASK 6: Models Without KNN Features ---")
print_model_latex(models_no_knn, "Baseline Models (No KNN Features)")

print("\n--- TASK 7: Models With KNN Features ---")
print_model_latex(models_with_knn, "Baseline Models (With KNN Features)")


--- TASK 6: Models Without KNN Features ---
\begin{table}[h]
\caption{Baseline Models (No KNN Features)}
\centering
\begin{tabular}{lcc}
\hline
Model & MAE & MAPE (\%) \\
\hline
KNN Avg & 54.506 & 5.883 \\
LSTM (No KNN) & 56.180 & 6.167 \\
DNN (No KNN) & 53.872 & 5.729 \\
ARIMA (No KNN) & 66.602 & 7.372 \\
\hline
\end{tabular}
\end{table}


--- TASK 7: Models With KNN Features ---
\begin{table}[h]
\caption{Baseline Models (With KNN Features)}
\centering
\begin{tabular}{lcc}
\hline
Model & MAE & MAPE (\%) \\
\hline
KNN Avg & 54.506 & 5.883 \\
LSTM (With KNN) & 53.766 & 5.869 \\
DNN (With KNN) & 51.944 & 5.563 \\
ARIMA (With KNN) & 57.823 & 6.097 \\
\hline
\end{tabular}
\end{table}



In [ ]:


# ==========================================
# TASK 8 & 9: Macro Models MAPE vs Threshold
# ==========================================
plot_thresholds(models_no_knn, 'mape', f"Task8_Baselines_NoKNN_MAPE_{station}.png")
plot_thresholds(models_with_knn, 'mape', f"Task9_Baselines_WithKNN_MAPE_{station}.png")

# ==========================================
# TASK 10 & 11: Scatter KNNC2 vs CQR
# ==========================================
save_scatter(vars_['knnc2_var'], vars_['cqr_var_no_knn'], "KNNC2 Pooled Variance", "CQR Variance (No KNN)", f"Task10_Scatter_KNNC2_CQR_NoKNN_{station}.png")
save_scatter(vars_['knnc2_var'], vars_['cqr_var_with_knn'], "KNNC2 Pooled Variance", "CQR Variance (With KNN)", f"Task11_Scatter_KNNC2_CQR_WithKNN_{station}.png")

# ==========================================
# TASK 12 & 13: Correlation Matrices
# ==========================================
def create_corr_matrix(data_dict, title):
    df = pd.DataFrame(data_dict)
    corr = df.corr()
    print(f"\n--- {title} ---")
    print(corr.to_markdown())
    return corr

t12_data = {
    'KNNC2': vars_['knnc2_var'],
    'CQR (No KNN)': vars_['cqr_var_no_knn'],
    'SPCI (No KNN)': vars_['spci_var_no_knn'],
    'Rolling Var': vars_['rolling_var']
}
create_corr_matrix(t12_data, "TASK 12: Correlation Matrix (Without KNN Features)")

t13_data = {
    'KNNC2': vars_['knnc2_var'],
    'CQR (With KNN)': vars_['cqr_var_with_knn'],
    'SPCI (With KNN)': vars_['spci_var_with_knn'],
    'Rolling Var': vars_['rolling_var']
}
create_corr_matrix(t13_data, "TASK 13: Correlation Matrix (With KNN Features)")


--- TASK 12: Correlation Matrix (Without KNN Features) ---
|               |    KNNC2 |   CQR (No KNN) |   SPCI (No KNN) |   Rolling Var |
|:--------------|---------:|---------------:|----------------:|--------------:|
| KNNC2         | 1        |       0.588079 |        0.609777 |      0.215005 |
| CQR (No KNN)  | 0.588079 |       1        |        0.886308 |      0.381049 |
| SPCI (No KNN) | 0.609777 |       0.886308 |        1        |      0.3574   |
| Rolling Var   | 0.215005 |       0.381049 |        0.3574   |      1        |

--- TASK 13: Correlation Matrix (With KNN Features) ---
|                 |    KNNC2 |   CQR (With KNN) |   SPCI (With KNN) |   Rolling Var |
|:----------------|---------:|-----------------:|------------------:|--------------:|
| KNNC2           | 1        |         0.662058 |          0.662058 |      0.215005 |
| CQR (With KNN)  | 0.662058 |         1        |          1        |      0.306286 |
| SPCI (With KNN) | 0.662058 |         1        |          

,KNNC2,CQR (With KNN),SPCI (With KNN),Rolling Var
KNNC2,1.000000,0.662058,0.662058,0.215005
CQR (With KNN),0.662058,1.000000,1.000000,0.306286
SPCI (With KNN),0.662058,1.000000,1.000000,0.306286
Rolling Var,0.215005,0.306286,0.306286,1.000000


In [ ]:
import pickle
import os
import numpy as np
import pandas as pd
from scipy.stats import norm
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

# ==============================================================================
# TASK 14: RANK-BASED PREDICTIVE ACCURACY TEST COMPUTATION
# ==============================================================================
def compute_loss_differential_results(
    timestamps, y_true, y_pred, disp_m1, disp_m2,
    alphas=[0.01, 0.02, 0.05, 0.10, 0.15, 0.20],
    hac_lags=None,
    metadata=None
):
    """
    Computes the selected-loss differentials and Z-statistics across various alpha levels[cite: 8].
    """
    loss_t = np.abs(y_true - y_pred)
    T = len(loss_t)

    if hac_lags is None:
        hac_lags = int(4 * (T / 100)**(2/9))

    z_stats_list, p_vals_list, d_bar_list, hac_se_list = [], [], [], []
    fixed_alpha_05_data = {}

    for alpha in alphas:
        thresh_m1 = np.quantile(disp_m1, 1 - alpha)
        thresh_m2 = np.quantile(disp_m2, 1 - alpha)

        A_t_1 = (disp_m1 >= thresh_m1).astype(int)
        A_t_2 = (disp_m2 >= thresh_m2).astype(int)

        d_t = loss_t * (A_t_1 - A_t_2)
        d_bar = np.mean(d_t)

        gamma_0 = np.mean((d_t - d_bar)**2)
        hac_var = gamma_0
        for j in range(1, hac_lags + 1):
            gamma_j = np.mean((d_t[j:] - d_bar) * (d_t[:-j] - d_bar))
            weight = 1 - (j / (hac_lags + 1))
            hac_var += 2 * weight * gamma_j

        hac_se = np.sqrt(hac_var / T) if hac_var > 0 else 1e-9
        z_stat = d_bar / hac_se
        p_val = 2 * (1 - norm.cdf(abs(z_stat)))

        z_stats_list.append(z_stat)
        p_vals_list.append(p_val)
        d_bar_list.append(d_bar)
        hac_se_list.append(hac_se)

        if np.isclose(alpha, 0.05):
            mean_loss_u1 = np.mean(loss_t[A_t_1 == 1]) if sum(A_t_1) > 0 else 0
            mean_loss_u2 = np.mean(loss_t[A_t_2 == 1]) if sum(A_t_2) > 0 else 0
            fixed_alpha_05_data = {
                'd_t': d_t,
                'cum_d_t': np.cumsum(d_t),
                'loss_m1': mean_loss_u1,
                'loss_m2': mean_loss_u2,
            }

    if not fixed_alpha_05_data and len(alphas) > 0:
        fixed_alpha_05_data = {
            'd_t': d_t, 'cum_d_t': np.cumsum(d_t), 'loss_m1': 0, 'loss_m2': 0
        }

    if metadata is None:
        metadata = {'m1_label': 'Method 1', 'm2_label': 'Method 2', 'dataset_name': 'Test Dataset'}

    return {
        'timestamps': timestamps,
        'alphas': alphas,
        'fixed_alpha_05': fixed_alpha_05_data,
        'sensitivity': {
            'z_statistics': z_stats_list,
            'p_values': p_vals_list,
            'd_bar_means': d_bar_list,
            'hac_errors': hac_se_list
        },
        'metadata': metadata
    }


# ==============================================================================
# TASK 14: RANK-BASED PREDICTIVE ACCURACY TEST PLOTTING
# ==============================================================================
def plot_loss_differential_results(results_dict):
    """Generates a high-res publication-grade standalone layout for the rank-based test sensitivity analysis[cite: 7]."""
    plt.style.use('seaborn-v0_8-whitegrid')

    # Adjusted to a single 1x1 subplot with appropriate dimensions[cite: 7]
    fig, ax = plt.subplots(1, 1, figsize=(8, 6))
    meta = results_dict['metadata']


    # --- Alpha Sensitivity Analysis ---
    alphas = results_dict['alphas']
    z_stats = results_dict['sensitivity']['z_statistics']

    ax.plot(alphas, z_stats, marker='o', color='purple', linewidth=2, markersize=7, label='Test Statistic (Z)')

    # Red dashed critical bounds[cite: 7]
    ax.axhline(y=1.96, color='red', linestyle='--', linewidth=1.5, label='Significance Level (5%)')
    ax.axhline(y=-1.96, color='red', linestyle='--', linewidth=1.5)

    # Shade significant regions (Values outside [-1.96, 1.96])[cite: 7]
    ax.fill_between(alphas, 1.96, max(max(z_stats), 3), color='green', alpha=0.1)
    ax.fill_between(alphas, min(min(z_stats), -3), -1.96, color='green', alpha=0.1)

    ax.set_title(f"Test Sensitivity Across Thresholds ($\\alpha$)\nNull: $\\Delta_\\alpha = 0$\n{meta['dataset_name']}\n1st dispersion model: {meta['m1_label']}\n2nd dispersion model: {meta['m2_label']}", fontsize=13, fontweight='bold')
    ax.set_xlabel("Threshold Level ($\\alpha$)", fontsize=11)
    ax.set_ylabel("Z-Statistic", fontsize=11)
    ax.tick_params(axis='both', labelsize=10)
    ax.legend(loc='best')

    plt.tight_layout()
    return fig, ax


# ==============================================================================
# 3. TASK 14: EXECUTION WRAPPER
# ==============================================================================
def execute_task14_for_models(data_dict, pred_model_key, disp_m1_key, disp_m2_key, station_id, save_dir_path):
    """
    Extracts relevant arrays from the reloaded data dictionary, computes the test,
    plots the sensitivity analysis, and saves the figure.
    """
    print(f"\nEvaluating: {pred_model_key} | {disp_m1_key} vs {disp_m2_key}")

    # Extract target and timestamp data
    y_true = data_dict['targets']['y_test']
    timestamps = data_dict['targets']['timestamps']

    # Extract prediction and variance data based on string keys
    y_pred = data_dict['predictions'][pred_model_key]
    disp_m1 = data_dict['variances'][disp_m1_key]
    disp_m2 = data_dict['variances'][disp_m2_key]

    # Build Metadata
    metadata = {
        'm1_label': disp_m1_key.replace('_', ' ').upper(),
        'm2_label': disp_m2_key.replace('_', ' ').upper(),
        'dataset_name': f"Station {station_id} - Model: {pred_model_key.replace('_', ' ').upper()}"
    }

    # Compute
    results = compute_loss_differential_results(
        timestamps=timestamps,
        y_true=y_true,
        y_pred=y_pred,
        disp_m1=disp_m1,
        disp_m2=disp_m2,
        metadata=metadata
    )

    # Plot
    fig, ax = plot_loss_differential_results(results)

    # Save
    filename = f"Task14_RankTest_Station{station_id}_{pred_model_key}_{disp_m1_key}_VS_{disp_m2_key}.png"
    save_path = os.path.join(save_dir_path, filename)
    fig.savefig(save_path, dpi=300)
    plt.close(fig)

    print(f"-> Saved plot to: {filename}")

In [ ]:
# ==============================================================================
# 4. RUN EXPERIMENTS
# ==============================================================================
print(f"{'='*50}\nRUNNING DYNAMIC RANK-BASED PREDICTIVE ACCURACY TESTS\n{'='*50}")

import itertools

pred_model_key_list_no_knn = ['lstm_no_knn', 'dnn_no_knn', 'arima_no_knn']
#pred_model_key_list_with_knn = ['lstm_with_knn', 'dnn_with_knn', 'arima_with_knn']
disp_key_list = ['knnc2_var', 'cqr_var_no_knn', 'spci_var_no_knn', 'rolling_var']
#disp_m2_key_list = ['knnc2_var', 'cqr_var_with_knn', 'spci_var_with_knn', 'rolling_var']

# Get all two-item combinations
combinations = list(itertools.combinations(disp_key_list, 2))

for combo in combinations:
    for pred_model_k in pred_model_key_list_no_knn:
        disp_m1_key = combo[0]
        disp_m2_key = combo[1]
        execute_task14_for_models(
            data_dict=data,
            pred_model_key=pred_model_k,
            disp_m1_key=disp_m1_key,
            disp_m2_key=disp_m2_key,
            station_id = st_num,
            save_dir_path=save_dir_after_reload
            )


print("\nAll dynamic Task 14 comparisons completed successfully.")

RUNNING DYNAMIC RANK-BASED PREDICTIVE ACCURACY TESTS

Evaluating: lstm_no_knn | knnc2_var vs cqr_var_no_knn
-> Saved plot to: Task14_RankTest_Station716241_lstm_no_knn_knnc2_var_VS_cqr_var_no_knn.png

Evaluating: dnn_no_knn | knnc2_var vs cqr_var_no_knn
-> Saved plot to: Task14_RankTest_Station716241_dnn_no_knn_knnc2_var_VS_cqr_var_no_knn.png

Evaluating: arima_no_knn | knnc2_var vs cqr_var_no_knn
-> Saved plot to: Task14_RankTest_Station716241_arima_no_knn_knnc2_var_VS_cqr_var_no_knn.png

Evaluating: lstm_no_knn | knnc2_var vs spci_var_no_knn
-> Saved plot to: Task14_RankTest_Station716241_lstm_no_knn_knnc2_var_VS_spci_var_no_knn.png

Evaluating: dnn_no_knn | knnc2_var vs spci_var_no_knn
-> Saved plot to: Task14_RankTest_Station716241_dnn_no_knn_knnc2_var_VS_spci_var_no_knn.png

Evaluating: arima_no_knn | knnc2_var vs spci_var_no_knn
-> Saved plot to: Task14_RankTest_Station716241_arima_no_knn_knnc2_var_VS_spci_var_no_knn.png

Evaluating: lstm_no_knn | knnc2_var vs rolling_var
-> Save